# Colab GPU cinematic test — Wan2.2 TI2V-5B

این نوت‌بوک برای **آزمایش جداگانه** است و به فایل‌های اصلی پروژه دست نمی‌زند. ابتدا GPU و فضای RAM/دیسک را بررسی می‌کند. تولید کامل فقط در صورت شناسایی حداقل 24GB حافظه GPU فعال می‌شود؛ در غیر این صورت دانلود سنگین مدل را شروع نمی‌کند.

**اجرا:** در Google Colab از Runtime → Change runtime type → GPU را انتخاب کنید، سپس سلول‌ها را به ترتیب اجرا کنید. این آزمایش تضمین نمی‌کند که هر نوع GPU کولب برای مدل مناسب باشد.

In [ ]:
import os, shutil, platform
print('Python:', platform.python_version())
print('Disk free (GB):', round(shutil.disk_usage('/content').free / 1024**3, 1))
try:
    import psutil
    print('System RAM (GB):', round(psutil.virtual_memory().total / 1024**3, 1))
except Exception as e:
    print('RAM check unavailable:', e)
try:
    import torch
    print('PyTorch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        vram_gb = props.total_memory / 1024**3
        print('GPU:', props.name)
        print('VRAM (GB):', round(vram_gb, 1))
        print('CUDA:', torch.version.cuda)
        if vram_gb >= 24:
            print('RESULT: GPU meets the model’s published 24GB baseline; generation can be attempted.')
        else:
            print('RESULT: Below 24GB. Do not attempt the official Wan2.2 TI2V-5B configuration in this notebook.')
    else:
        print('RESULT: No CUDA GPU detected. Change Colab runtime to GPU and rerun.')
except Exception as e:
    print('GPU diagnostic failed:', repr(e))

## Optional: install the official engine and download its weights

Run this only if the diagnostic above reports at least 24GB VRAM and enough free disk. The model download is large and may take a while. Colab availability and runtime duration are not guaranteed.

In [ ]:
import os, subprocess, torch, shutil
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU detected. Stop here; select a GPU runtime.')
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
if vram_gb < 24:
    raise RuntimeError(f'GPU has {vram_gb:.1f}GB VRAM; this test requires at least 24GB for the official configuration. No model download started.')
if shutil.disk_usage('/content').free < 35 * 1024**3:
    raise RuntimeError('Less than 35GB free disk space. Free space or use a larger runtime before downloading.')
if not os.path.isdir('/content/Wan2.2'):
    subprocess.run(['git','clone','https://github.com/Wan-Video/Wan2.2.git','/content/Wan2.2'], check=True)
%cd /content/Wan2.2
subprocess.run(['pip','install','-r','requirements.txt'], check=True)
subprocess.run(['pip','install','huggingface_hub'], check=True)
from huggingface_hub import snapshot_download
snapshot_download(repo_id='Wan-AI/Wan2.2-TI2V-5B', local_dir='/content/Wan2.2/Wan2.2-TI2V-5B')
print('Engine and weights downloaded.')

## Generate one short test clip

This cell creates a short proof-of-concept clip, not a finished movie scene. If this cell fails, save the complete error output so the configuration can be corrected without repeatedly downloading the model.

In [ ]:
import os, subprocess, torch
if not torch.cuda.is_available() or torch.cuda.get_device_properties(0).total_memory / 1024**3 < 24:
    raise RuntimeError('GPU does not meet the 24GB baseline; generation was not started.')
os.chdir('/content/Wan2.2')
prompt = ('Cinematic wide shot at dusk in an old Persian courtyard, two adult characters having a serious emotional conversation, '
          'subtle natural gestures, consistent faces, realistic fabric and skin, warm practical lantern lighting, slow camera dolly, '
          'film color grading, no text, no subtitles.')
cmd = ['python','generate.py','--task','ti2v-5B','--size','1280*704','--ckpt_dir','./Wan2.2-TI2V-5B',
       '--offload_model','True','--convert_model_dtype','--t5_cpu','--prompt',prompt]
print('Starting one test generation...')
subprocess.run(cmd, check=True)
print('Check /content/Wan2.2 for the generated output files.')